### Required dependencies

In [2]:
import os
import fitz  # PyMuPDF
from ocr_tamil.ocr import OCR 


### Initialize OCR model

In [3]:
ocr_detect_tamil = OCR(
    detect=True,
    lang=["tamil", "english"],
    batch_size=128,
    tamil_model_path=r"../model_weights/parseq_tamil_v3.pt",
    eng_model_path=r"../model_weights/craft_mlt_25k.pth"
)

saving to /home/arunachalam/.model_weights/craft_mlt_25k.pth
Download would take several minutes


100%|██████████| 83.2M/83.2M [01:07<00:00, 1.24MB/s]


### Parsing the FIR file (PDF -> TEXT)

##### This code reads a PDF file page-by-page using PyMuPDF (fitz), converts each page into an image, and performs OCR using ocr_tamil a custom Tamil+English OCR model. For every page image, the model extracts text, which is then formatted into clean lines (6 words per line) for readability.

##### After processing all pages, the extracted text is combined into a single output, printed for review, and finally saved as a .txt file. Temporary images created during processing are automatically deleted to keep your workspace clean.

In [19]:

# PDF path
pdf_path = "../data/CBCID-Salem Cr.No.01-2017 Acquital case.pdf"

# Open PDF
pdf_document = fitz.open(pdf_path)

all_texts = []
for page_num in range(len(pdf_document)):
    page = pdf_document.load_page(page_num)
    pix = page.get_pixmap()
    image_path = f"../data/temp_page_{page_num}.png"
    pix.save(image_path)

    try:
        # OCR the temp image
        texts = ocr_detect_tamil.predict(image_path)
        full_text = " ".join(texts[0])

        # Format text: 6 words per line
        words = full_text.split()
        formatted_text = "\n".join(
            [" ".join(words[i:i+6]) for i in range(0, len(words), 6)]
        )
        all_texts.append(formatted_text)

    finally:
        # Delete temp file safely
        if os.path.exists(image_path):
            os.remove(image_path)

# Combine all pages
final_text = "\n\n".join(all_texts)

# Save extracted text to file
output_file_path = "../data/output_text_Cr.No.01-2017.txt"
with open(output_file_path, "w", encoding="utf-8") as f:
    f.write(final_text)

print(f"Final text saved to {output_file_path}")


Final text saved to ../data/output_text_Cr.No.01-2017.txt


###  Process All PDFs in Folder (Supports Subfolders Automatically)
This function scans the input directory recursively, including all zone subfolders  
(CZ, SZ, WZ, NZ), and creates the **same folder structure** inside the output directory.

**Features:**
- Automatically preserves subfolder hierarchy  
- Creates missing output directories  
- Processes every PDF inside each subfolder  
- Saves each output as `<filename>_parsed.txt` inside the matching zone folder  


In [17]:
# -----------------------------
# PROCESS FOLDER WITH SUBFOLDERS
# -----------------------------
def process_pdf_folder(input_folder: str, output_folder: str):
    for root, dirs, files in os.walk(input_folder):
        sub_path = os.path.relpath(root, input_folder)

        # Create matching output folder
        out_subfolder = os.path.join(output_folder, sub_path)
        os.makedirs(out_subfolder, exist_ok=True)

        # Process PDFs inside this subfolder
        for file in files:
            if file.lower().endswith(".pdf"):
                print(f"Processing: {file} in {sub_path}")

                pdf_path = os.path.join(root, file)
                parsed_text = parse_single_pdf(pdf_path)

                output_name = os.path.splitext(file)[0] + "_parsed.txt"
                output_path = os.path.join(out_subfolder, output_name)

                with open(output_path, "w", encoding="utf-8") as f:
                    f.write(parsed_text)

                print(f"Saved → {output_path}\n")


###  Run the PDF Parsing Script  
Use the **parent input folder** if you want the output to automatically preserve the full zone structure (CZ/SZ/WZ/NZ).  
You may also pass a **single subfolder (e.g., CZ)** if you want to process only one zone.


In [ ]:
# Set paths here (use parent folder for full structure, or subfolder to process only that zone)
input_folder = "../data/FIRs_input_folder/CZ"        # or "../data/FIRs_input_folder/CZ"
output_folder = "../data/Parsed_FIRs_output_folder"   

process_pdf_folder(input_folder, output_folder)